# Small Hindi → English Transformer (PyTorch, from scratch)

An encoder–decoder transformer written one function / class per job.
Run the cells **top to bottom**. Every section has: what it does, the code (block comment + line comments), and a quick sanity test.

**Pipeline**

`text file → split → tokens → ids → padded batches → masks → embeddings + positions → attention → encoder → decoder → logits → loss → backward → Adam → epochs → generate`

**Before you start:** put `hindi_english.txt` (one `hindi<TAB>english` pair per line) in the same folder as this notebook.

**How the comments are written**

* `# ==== name ====` block above a function / class: **WHAT** it does, **WHY** it exists, and the **SHAPES** of its inputs and outputs.
* `# ...` at the end of a line: what that single line does, especially when a PyTorch call hides a lot of work.
* Shapes use `B` = batch size, `L` / `T` / `S` = sequence length (source `S`, target `T`), `H` = heads, `d_k` = size per head, `V` = vocabulary size.

## PyTorch cheat sheet (the calls used in this notebook)

| Call | What it does |
|---|---|
| `x.shape`, `x.size(-1)` | The dimensions of a tensor. `-1` means "the last dimension". |
| `x.unsqueeze(i)` | Inserts a new dimension of size 1 at position `i`. Used so shapes can **broadcast**. |
| `x.view(...)` / `x.reshape(...)` | Same numbers, new shape. The total count of numbers must stay the same. `-1` = "work this size out for me". |
| `x.transpose(a, b)` | Swaps dimensions `a` and `b`. |
| `x.contiguous()` | Re-lays the numbers out in memory in order. `view` needs this after a `transpose`. |
| `a @ b` / `torch.matmul(a, b)` | Matrix multiply on the **last two** dimensions; any earlier dimensions (batch, heads) are carried along. |
| `x.masked_fill(mask, v)` | Puts the value `v` wherever `mask` is True. |
| `F.softmax(x, dim=-1)` | Turns scores into probabilities that sum to 1 along `dim`. |
| `torch.tril(m)` | Keeps the lower triangle of a matrix, zeroes the rest. |
| `torch.cat([a, b], dim=1)` | Joins tensors along an existing dimension. |
| `x.argmax(dim=-1)` | Position (index) of the largest value. |
| **Broadcasting** | When two tensors meet in an operation, a dimension of size 1 is stretched to match the other's size. |
| `nn.Module` | Base class of every layer / model. Anything assigned in `__init__` as a layer is tracked, and its weights are trained. |
| `nn.Linear(a, b)` | A learnable matrix: maps the last dimension from size `a` to size `b` (`x @ W + bias`). |
| `nn.Embedding(V, d)` | A learnable table of `V` rows (one per token), each a vector of `d` numbers. |
| `nn.LayerNorm(d)` | Rescales each token's `d` numbers to mean 0 / std 1, then applies a learnable scale and shift. |
| `nn.Dropout(p)` | While training, randomly zeroes a fraction `p` of values (stops memorising). Does nothing in `eval()` mode. |
| `nn.ModuleList` | A list of layers that PyTorch can see (a plain Python list would hide them, and they would never train). |
| `register_buffer` | A tensor that belongs to the model (saved, moved to GPU) but is **not** a trainable weight. |
| `model.train()` / `model.eval()` | Switch dropout on / off. |
| `torch.no_grad()` | Do not record gradients inside this block: faster, uses less memory. Used when only measuring or generating. |
| `loss.backward()` | Computes the gradient of the loss for every weight. |
| `optimizer.zero_grad()` / `.step()` | Clear old gradients / update every weight using the gradients. |
| `model.state_dict()` | All learned weights as a dictionary: what you save to disk. |

## 0. Imports and config

All hyperparameters live in the single `config` dict. To train on a bigger file, change `data_path`
(and set `retrain` to `True` so the old checkpoint is not reused).

In [1]:
# ==== Imports ===================================================================
# WHAT: loads the libraries the rest of the notebook needs.
import math                      # math.sqrt, math.log -> scaling and the positional-encoding formula
import os                        # os.path.exists -> "is there already a saved checkpoint?"
import random                    # shuffling the data in a repeatable way
import re                        # regular expressions -> put spaces around the danda "।"

import torch                     # the core library: tensors (n-dimensional arrays) + autograd (automatic gradients)
import torch.nn as nn            # nn = layers with weights: Linear, Embedding, LayerNorm, Dropout ...
import torch.nn.functional as F  # F  = functions with no weights inside: F.softmax, F.relu

In [2]:
# ==== Config ====================================================================
# WHAT: every hyperparameter and file path in one dict.
# WHY : to run an experiment you change a value HERE, not code scattered around the notebook.
config = {
    # ---- data / files
    "data_path": "hindi_english.txt",     # tab-separated: hindi <TAB> english, one pair per line
    "checkpoint_path": "best_model.pt",   # the best weights + vocab + config are saved in this file
    "retrain": True,                      # False -> reuse the checkpoint if it exists; True -> always train again
    "seed": 42,                           # same seed -> same data split and same starting weights

    # ---- model size
    "d_model": 64,       # length of the vector that represents each token (bigger = more capacity)
    "num_heads": 4,      # attention heads; d_model must divide evenly by this (64 / 4 = 16 per head)
    "d_ff": 128,         # hidden size inside the feed-forward layer (usually 2-4x d_model)
    "num_layers": 2,     # number of encoder layers AND number of decoder layers
    "dropout": 0.1,      # fraction of values randomly zeroed during training (regularisation)
    "max_len": 100,      # longest sequence (in tokens) the positional encoding table supports

    # ---- training
    "lr": 5e-4,          # learning rate: how big each Adam weight update is (too high = unstable, too low = slow)
    "batch_size": 64,    # sentences processed together in one training step
    "num_epochs": 20,     # one epoch = one full pass over the training set
}

random.seed(config["seed"])          # python's shuffle becomes repeatable
torch.manual_seed(config["seed"])    # the random initial weights (and dropout masks) become repeatable

## 1. Data: load and split

* **train (80%)**: the model learns from these; weights are updated on them.
* **validation (10%)**: not used for updates; checked after every epoch to spot overfitting.
* **test (10%)**: touched once at the very end for an honest final score.

In [3]:
# ==== load_pairs_from_txt =======================================================
# WHAT : reads the text file -> list of (hindi, english) tuples.
# WHY  : every later step starts from this list.
# OUT  : [("hindi sentence", "english sentence"), ...]
def load_pairs_from_txt(file_path):
    pairs = []                                            # will collect the (hindi, english) tuples
    with open(file_path, 'r', encoding='utf-8') as f:     # 'with' closes the file for us; utf-8 is REQUIRED for Hindi letters
        for line in f:                                    # file objects hand out one line at a time
            line = line.strip()                           # remove the newline and any spaces at both ends
            if line:                                      # an empty string counts as False -> blank lines are skipped
                hindi, english = line.split("\t")         # cut at the TAB -> exactly 2 pieces, unpacked into 2 variables
                pairs.append((hindi, english))            # keep the two sentences together as one tuple
    return pairs


# ==== split_data_test_train =====================================================
# WHAT : shuffles the pairs once, then cuts the list into train / validation / test.
# WHY  : the model must be judged on sentences it never trained on.
# NOTE : the same seed always gives the same split, so results are reproducible.
def split_data_test_train(pairs, train_frac=0.8, val_frac=0.1, seed=42):
    shuffled = pairs[:]                          # [:] makes a COPY, so the caller's list keeps its order
    random.Random(seed).shuffle(shuffled)        # a private random generator with this seed; shuffle reorders in place

    n = len(shuffled)                            # total number of pairs
    train_size = int(n * train_frac)             # 10000 * 0.8 = 8000 (int() drops any decimals)
    val_size = int(n * val_frac)                 # 10000 * 0.1 = 1000

    train_pairs = shuffled[:train_size]                              # first 80%
    val_pairs = shuffled[train_size:train_size + val_size]           # next 10%
    test_pairs = shuffled[train_size + val_size:]                    # everything that is left (last 10%)
    return train_pairs, val_pairs, test_pairs

In [4]:
# ---- Test: load + split --------------------------------------------------------
pairs_data = load_pairs_from_txt(config["data_path"])
print(len(pairs_data))                      # expect 10000
print(pairs_data[0])                        # expect one (hindi, english) tuple

train_pairs, val_pairs, test_pairs = split_data_test_train(pairs_data, seed=config["seed"])
print("Train pairs     :", len(train_pairs))        # expect 8000
print("Validation pairs:", len(val_pairs))          # expect 1000
print("Test pairs      :", len(test_pairs))         # expect 1000

# set(...) & set(...) = the pairs present in BOTH splits. It must be empty (no leakage).
print("overlap train/val :", len(set(train_pairs) & set(val_pairs)))     # expect 0
print("overlap train/test:", len(set(train_pairs) & set(test_pairs)))    # expect 0

100000
('आदमी रात को दफ़्तर में फूल बेचता है ।', 'The man sells flowers in the office at night.')
Train pairs     : 80000
Validation pairs: 10000
Test pairs      : 10000
overlap train/val : 0
overlap train/test: 0


## 2. Tokenizing and vocabulary

A network only does maths on numbers, so text becomes **token ids**. A *vocabulary* is the lookup table `token ↔ id`.

* Word-level tokens (split on spaces) keep things simple.
* Special tokens: `<PAD>` (0) fills short sentences, `<SOS>` starts a sentence, `<EOS>` ends it, `<UNK>` replaces unseen words.
* Hindi and English each get their own vocab, built from the **train split only** (no peeking at val/test).
* `tokenize_text_line` puts a space around the danda `।`, so `है।` and `है ।` give the same tokens.

In [5]:
# ==== tokenize_text_line ========================================================
# WHAT : sentence (str) -> list of word tokens.
# WHY  : the model sees words as units; the danda must be its own token.
# EX   : "लड़की खाती है।"  ->  ['लड़की', 'खाती', 'है', '।']
def tokenize_text_line(sentence):
    # regex: \s* = "zero or more whitespace characters", then the danda, then \s* again.
    # Whatever spaces surround the danda are replaced by exactly one space on each side.
    sentence = re.sub(r"([।?!,.])", r" \1 ", sentence)
    return sentence.split()                       # split() with no argument cuts at any run of spaces


# ==== build_vocab_from_sentences ================================================
# WHAT : collects every unique word and numbers them -> the vocabulary.
# OUT  : stoi = dict  token -> id   (used to ENCODE text)
#        itos = list  id -> token   (used to DECODE ids; list position = id)
def build_vocab_from_sentences(sentences):
    special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']          # get ids 0, 1, 2, 3 (PAD must be 0)
    # {...} with a for loop is a SET comprehension: a set keeps each word only once.
    # sorted() gives a fixed order, so the same data always gives the same ids.
    all_tokens = sorted({tok for s in sentences for tok in tokenize_text_line(s)})
    itos = special_tokens + all_tokens                             # specials first, then the real words
    stoi = {tok: i for i, tok in enumerate(itos)}                  # enumerate gives (0, tok0), (1, tok1)... -> token: id
    return stoi, itos


# ==== encode_sentence ===========================================================
# WHAT : text -> list of ids, wrapped in <SOS> ... <EOS>.
# NOTE : a word that is not in the vocabulary becomes <UNK> instead of crashing.
def encode_sentence(sentence, stoi):
    tokens = tokenize_text_line(sentence)                          # words
    encoded = [stoi.get(token, stoi['<UNK>']) for token in tokens] # dict.get(key, default): the id if known, else the <UNK> id
    return [stoi['<SOS>']] + encoded + [stoi['<EOS>']]             # add the start and end markers


# ==== decode_sequence ===========================================================
# WHAT : list of ids -> text (the reverse of encode). <PAD>/<SOS>/<EOS> are dropped.
def decode_sequence(encoded_ids, itos):
    skip = ['<PAD>', '<SOS>', '<EOS>']                             # markers that are not real words
    # keep an id only if it is a valid position in itos AND its word is not a marker
    words = [itos[i] for i in encoded_ids if i < len(itos) and itos[i] not in skip]
    return re.sub(r"\s+([,.?!])", r"\1", ' '.join(words))         # join with spaces, then remove the space before , . ? !

In [6]:
# ---- Test: vocab + encode/decode round trip ------------------------------------
src_stoi, src_itos = build_vocab_from_sentences([p[0] for p in train_pairs])   # Hindi vocab  (p[0] = Hindi side)
tgt_stoi, tgt_itos = build_vocab_from_sentences([p[1] for p in train_pairs])   # English vocab (p[1] = English side)
print(len(src_stoi), len(tgt_stoi))                    # expect 77 98 (both include the 4 special tokens)

ids = encode_sentence(train_pairs[0][1], tgt_stoi)
print(ids)                                             # starts with 1 (<SOS>), ends with 2 (<EOS>)
print(decode_sequence(ids, tgt_itos))                  # must print the original English sentence

# the danda fix: attached or spaced must give the SAME tokens
print(tokenize_text_line("लड़की रोज़ आम खाती है।"))
print(tokenize_text_line("लड़की रोज़ आम खाती है ।"))

451 450
[1, 38, 168, 357, 47, 77, 214, 403, 191, 214, 403, 274, 5, 2]
The farmer sees a bird in the garden in the morning.
['लड़की', 'रोज़', 'आम', 'खाती', 'है', '।']
['लड़की', 'रोज़', 'आम', 'खाती', 'है', '।']


## 3. Batching: padding and the decoder shift

Training works on many sentences at once as one rectangular tensor `[batch, length]`, so short sentences are padded with `<PAD>` (0).

**Teacher forcing** - the decoder gets the correct English sentence shifted by one:

```
tgt_in  = <SOS> They sell milk        (what the decoder reads)
tgt_out = They  sell milk <EOS>       (what it must predict at each position)
```

Slice **before** padding, so `<EOS>` never ends up inside `tgt_in`.

In [7]:
# ==== pad_sequences =============================================================
# WHAT : makes sentences of different lengths into one rectangle by appending <PAD> ids.
# IN   : list of id-lists, e.g. [[1,9,2], [1,9,78,57,2]]
# OUT  : LongTensor [B, longest_length]
def pad_sequences(batch_ids, pad_token_id):
    max_length = max(len(ids) for ids in batch_ids)             # length of the longest sentence in THIS batch
    padded = []
    for ids in batch_ids:
        n_pad = max_length - len(ids)                           # how many <PAD>s this sentence is missing
        padded.append(ids + [pad_token_id] * n_pad)             # [0] * 3 = [0, 0, 0]; list + list joins them
    # dtype=torch.long = 64-bit integers. nn.Embedding uses these ids as ROW NUMBERS, so they must be integers.
    return torch.tensor(padded, dtype=torch.long)               # list of equal-length lists -> 2-D tensor


# ==== make_batch ================================================================
# WHAT : turns a list of (hindi, english) pairs into the three tensors training needs.
# OUT  : src [B,S] (Hindi ids), tgt_in [B,T] (decoder input), tgt_out [B,T] (what the decoder must predict)
def make_batch(pairs, src_stoi, tgt_stoi):
    src_encoded = [encode_sentence(p[0], src_stoi) for p in pairs]     # Hindi sentences -> id lists
    tgt_encoded = [encode_sentence(p[1], tgt_stoi) for p in pairs]     # English sentences -> id lists

    tgt_in_encoded = [ids[:-1] for ids in tgt_encoded]     # [:-1] = everything except the LAST id  -> <SOS> w1 w2 ...
    tgt_out_encoded = [ids[1:] for ids in tgt_encoded]     # [1:]  = everything except the FIRST id -> w1 w2 ... <EOS>

    src_padded = pad_sequences(src_encoded, src_stoi['<PAD>'])
    tgt_in_padded = pad_sequences(tgt_in_encoded, tgt_stoi['<PAD>'])
    tgt_out_padded = pad_sequences(tgt_out_encoded, tgt_stoi['<PAD>'])
    return src_padded, tgt_in_padded, tgt_out_padded


# ==== get_batches ===============================================================
# WHAT : hands out the data one batch at a time.
# NOTE : 'yield' makes this a GENERATOR: it pauses after each batch and resumes on the next loop turn,
#        so only one batch of tensors exists at a time (memory-friendly for big data).
def get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):
    pairs = pairs[:]                       # copy, so shuffling does not reorder the caller's list
    if shuffle:
        random.shuffle(pairs)              # training: new order every epoch; validation / test: shuffle=False
    for i in range(0, len(pairs), batch_size):          # i = 0, 64, 128, ... (step = batch_size)
        batch_pairs = pairs[i:i + batch_size]           # the last batch may be smaller than batch_size
        yield make_batch(batch_pairs, src_stoi, tgt_stoi)

In [8]:
# ---- Test: padding, shift, shapes ----------------------------------------------
print(pad_sequences([[1, 9, 78, 2], [1, 9, 78, 57, 14, 2], [1, 9, 2]], 0))   # 3 rows, 6 columns, zeros at the end

# next(generator) = take just the first batch
src, tgt_in, tgt_out = next(get_batches(train_pairs, 64, src_stoi, tgt_stoi))
print(src.shape, tgt_in.shape, tgt_out.shape)        # first number 64; tgt_in and tgt_out have the SAME shape
print(tgt_in[0])                                     # starts with 1 (<SOS>)
print(tgt_out[0])                                    # the same ids moved one place left; 2 (<EOS>) after the last word

tensor([[ 1,  9, 78,  2,  0,  0],
        [ 1,  9, 78, 57, 14,  2],
        [ 1,  9,  2,  0,  0,  0]])
torch.Size([64, 21]) torch.Size([64, 16]) torch.Size([64, 16])
tensor([  1,  38, 398, 106, 434, 214, 403, 366, 291,  24,   5,   0,   0,   0,
          0,   0])
tensor([ 38, 398, 106, 434, 214, 403, 366, 291,  24,   5,   2,   0,   0,   0,
          0,   0])


## 4. Masks

A mask is a True/False grid saying which positions attention may look at (**True = allowed**).

* **Padding mask** - hides `<PAD>` filler. Shape `[B, 1, 1, L]`.
* **Causal mask** - decoder only; a word may not see later words (a lower-triangular matrix). Shape `[1, 1, T, T]`.
* **Target mask** = padding AND causal, so the decoder sees only real, earlier words.

The extra size-1 dimensions broadcast over the attention heads (and the batch for the causal mask).

In [9]:
# ==== make_pad_mask =============================================================
# WHAT : marks which positions are real tokens (True) and which are <PAD> filler (False).
# WHY  : attention must not "look at" padding.
# IN   : ids [B, L]        OUT: bool tensor [B, 1, 1, L]
def make_pad_mask(encoded_ids, pad_token_id):
    mask = (encoded_ids != pad_token_id)            # element-by-element comparison -> True where the id is NOT the pad id
    # [B, L] -> [B, 1, 1, L]: the two size-1 dimensions line up with (heads, query positions)
    # in the attention scores [B, H, Lq, Lk], so the mask stretches over all of them (broadcasting).
    return mask.unsqueeze(1).unsqueeze(2)


# ==== make_causal_mask ==========================================================
# WHAT : the "no peeking at the future" triangle for the decoder.
# EX (T=4): row = the word that is looking, column = the word being looked at
#           1 0 0 0
#           1 1 0 0
#           1 1 1 0
#           1 1 1 1
# OUT  : bool tensor [1, 1, T, T]
def make_causal_mask(encoded_ids):
    seq_len = encoded_ids.size(1)                                       # T = number of target positions
    # torch.ones(T, T) = a square of True values; torch.tril keeps the lower triangle (diagonal included), zeroes the rest.
    causal_mask = torch.tril(torch.ones((seq_len, seq_len), dtype=torch.bool))
    return causal_mask.unsqueeze(0).unsqueeze(1)                        # [T, T] -> [1, T, T] -> [1, 1, T, T]


# ==== make_tgt_mask =============================================================
# WHAT : decoder mask = (a real token) AND (not in the future).
# OUT  : bool tensor [B, 1, T, T]
def make_tgt_mask(encoded_ids, pad_token_id):
    pad_mask = make_pad_mask(encoded_ids, pad_token_id)       # [B, 1, 1, T]
    causal_mask = make_causal_mask(encoded_ids)               # [1, 1, T, T]
    return pad_mask & causal_mask     # & = element-wise AND; the size-1 dims broadcast -> [B, 1, T, T]

In [10]:
# ---- Test: masks ---------------------------------------------------------------
src_mask = make_pad_mask(src, src_stoi['<PAD>'])
print(src_mask.shape, src_mask.dtype)                    # expect [64, 1, 1, L] and torch.bool (NOT int64)

print(make_causal_mask(tgt_in[:, :4])[0, 0].int())       # the 4x4 lower triangle ([0, 0] drops the two size-1 dims)

tgt_mask = make_tgt_mask(tgt_in, tgt_stoi['<PAD>'])
print(tgt_mask.shape)                                    # expect [64, 1, L, L]
print(tgt_mask[0, 0].int())                              # a triangle that stops growing at the last real token

torch.Size([64, 1, 1, 21]) torch.bool
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
torch.Size([64, 1, 16, 16])
tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1,

## 5. Embedding and positional encoding

* **Embedding** - a learnable table `[vocab, d_model]`; token id -> its vector. Scaled by `sqrt(d_model)` as in the paper.
* **Positional encoding** - attention has no sense of word order, so a fixed sin/cos pattern for each position is added:
  `PE[pos, 2i] = sin(pos / 10000^(2i/d))`, `PE[pos, 2i+1] = cos(...)`. It is a *buffer*: saved with the model, never trained.

In [11]:
# ==== TokenEmbedding ============================================================
# WHAT : turns token ids into learnable vectors.
# WHY  : the id 9 means nothing numerically; a vector of 64 learned numbers can capture "meaning".
# IN   : ids [B, L] (integers)      OUT: [B, L, d_model] (floats)
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, pad_id):
        super().__init__()          # sets up nn.Module bookkeeping (needed so PyTorch can find our layers / weights)
        # nn.Embedding = a trainable matrix of shape [vocab_size, d_model]; "lookup of id 9" = "give me row 9".
        # padding_idx: the <PAD> row stays all zeros and is never updated.
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.d_model = d_model

    def forward(self, x):           # forward() runs when we call the object like a function: emb(ids)
        # multiply by sqrt(d_model) so the embedding is not drowned out by the positional values added next
        return self.embedding(x) * math.sqrt(self.d_model)


# ==== PositionalEncoding ========================================================
# WHAT : adds a fixed, position-dependent vector to every token vector.
# WHY  : attention treats the sentence as a bag of words; without positions "girl eats apple" = "apple eats girl".
# IN/OUT: [B, L, d_model] -> [B, L, d_model]
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                       # empty table: one row per position, one column per dimension
        position = torch.arange(max_len).unsqueeze(1)            # 0,1,2,...,max_len-1 as a column -> [max_len, 1]
        # div_term = 1 / 10000^(2i/d_model) for i = 0,1,2,... (computed as exp(-(2i) * ln(10000) / d_model),
        # which is numerically safer). arange(0, d_model, 2) = 0,2,4,... one value per sin/cos PAIR -> [d_model/2]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        # [max_len,1] * [d_model/2] broadcasts to [max_len, d_model/2]: every position times every frequency.
        pe[:, 0::2] = torch.sin(position * div_term)             # 0::2 = every second column starting at 0 (even) -> sin
        pe[:, 1::2] = torch.cos(position * div_term)             # 1::2 = every second column starting at 1 (odd)  -> cos
        pe = pe.unsqueeze(0)                                     # add a batch dimension -> [1, max_len, d_model]
        # a buffer is saved with the model and moves with it to a GPU, but is NOT a trainable weight
        self.register_buffer("pe", pe)

    def forward(self, x):                                        # x: [B, L, d_model]
        x = x + self.pe[:, :x.size(1)]       # take the first L rows of the table; [1,L,d] + [B,L,d] broadcasts over the batch
        return self.dropout(x)               # randomly zero a few values while training

In [12]:
# ---- Test: embedding + positions -----------------------------------------------
emb = TokenEmbedding(len(src_stoi), config["d_model"], src_stoi['<PAD>'])
pos_enc = PositionalEncoding(config["d_model"], config["max_len"], config["dropout"])

x = emb(src)
print(x.shape)                         # [64, L, 64]: every id became a 64-number vector
print(pos_enc(x).shape)                # same shape: positions are ADDED, not appended
print(pos_enc.pe[0, 0, :4])            # expect tensor([0., 1., 0., 1.]): position 0 = sin(0), cos(0), sin(0), cos(0)
print(pos_enc.pe[0, 1, :4])            # position 1 must look different from position 0

torch.Size([64, 21, 64])
torch.Size([64, 21, 64])
tensor([0., 1., 0., 1.])
tensor([0.8415, 0.5403, 0.6816, 0.7318])


## 6. Attention

**Scaled dot-product attention** - each word makes a *query*, every word offers a *key* and a *value*.
Match score = query · key; softmax turns scores into weights; the output is the weighted average of the values:

`Attention(Q, K, V) = softmax(Q·Kᵀ / sqrt(d_k)) · V`

Masked positions get a huge negative score before the softmax, so their weight becomes ~0.

**Multi-head attention** splits `d_model` into several heads that each attend differently, then joins them.
The same class does *self*-attention (`q = k = v`) and *cross*-attention (`q` from the decoder, `k = v` from the encoder).

In [13]:
# ==== scaled_dot_product_attention ==============================================
# WHAT : for each query word, decide how much to look at every key word, then mix the values accordingly.
# IN   : query [B,H,Lq,d_k], key [B,H,Lk,d_k], value [B,H,Lk,d_k], mask (True = allowed) or None
# OUT  : output [B,H,Lq,d_k]  and  attention_weights [B,H,Lq,Lk] (each row sums to 1)
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)                                    # size of one head's vector (16)
    # key.transpose(-2, -1) swaps the last two dims: [.., Lk, d_k] -> [.., d_k, Lk].
    # matmul then gives one score for EVERY (query word, key word) pair: [B, H, Lq, Lk].
    # Dividing by sqrt(d_k) stops the scores growing with d_k (huge scores would make softmax too "sharp" to learn from).
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # where mask is False (== 0: padding / future) put -1e9, so softmax turns that weight into ~0
        scores = scores.masked_fill(mask == 0, -1e9)

    # softmax over the LAST dim (the key words): each query's weights become probabilities summing to 1
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)         # weighted average of the value vectors: [B, H, Lq, d_k]
    return output, attention_weights


# ==== MultiHeadAttention ========================================================
# WHAT : runs attention several times in parallel on different slices (heads) of the vectors, then joins the results.
# WHY  : each head can learn a different relationship (e.g. subject-verb, verb-object).
# IN   : query [B,Lq,d_model], key/value [B,Lk,d_model]   OUT: [B,Lq,d_model]
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"   # 64 % 4 == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads                     # each head works on 64 / 4 = 16 numbers
        self.W_q = nn.Linear(d_model, d_model)              # learnable "turn input into queries"
        self.W_k = nn.Linear(d_model, d_model)              # learnable "turn input into keys"
        self.W_v = nn.Linear(d_model, d_model)              # learnable "turn input into values"
        self.W_o = nn.Linear(d_model, d_model)              # learnable "mix the joined heads together"

    # [B, L, d_model] -> [B, H, L, d_k]
    def split_heads(self, x):
        B, L, _ = x.size()                                  # _ = a value we do not need (d_model)
        # view: cut the last dim (64) into 4 heads x 16 -> [B, L, 4, 16]
        # transpose(1, 2): swap L and H -> [B, 4, L, 16], so attention runs per head
        return x.view(B, L, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, Lq, _ = query.size()
        q = self.split_heads(self.W_q(query))               # Linear first, then split into heads: [B, H, Lq, d_k]
        k = self.split_heads(self.W_k(key))                 # [B, H, Lk, d_k]
        v = self.split_heads(self.W_v(value))               # [B, H, Lk, d_k]
        out, weights = scaled_dot_product_attention(q, k, v, mask)    # out: [B, H, Lq, d_k]
        # put the heads back side by side: swap H and L back -> [B, Lq, H, d_k]. contiguous() re-lays the memory
        # in order (view() cannot be used right after transpose without it); view merges H * d_k -> d_model.
        out = out.transpose(1, 2).contiguous().view(B, Lq, self.num_heads * self.d_k)
        return self.W_o(out)                                # final learnable mix: [B, Lq, d_model]

In [14]:
# ---- Test: attention -----------------------------------------------------------
q = torch.randn(2, 4, 5, 16); k = torch.randn(2, 4, 5, 16); v = torch.randn(2, 4, 5, 16)   # random [B,H,L,d_k]

out, w = scaled_dot_product_attention(q, k, v)
print(out.shape, w.shape)                      # expect [2,4,5,16] and [2,4,5,5]
print(w.sum(dim=-1)[0, 0])                     # all ones: every row of weights sums to 1

causal = make_causal_mask(torch.zeros(2, 5))   # a [1,1,5,5] triangle (the values of the dummy input do not matter)
out, w = scaled_dot_product_attention(q, k, v, causal)
print(w[0, 0])                                 # zeros ABOVE the diagonal; the first row must be [1, 0, 0, 0, 0]

mha = MultiHeadAttention(config["d_model"], config["num_heads"])
xx = torch.randn(2, 5, 64)
print(mha(xx, xx, xx).shape)                                       # self-attention: [2, 5, 64]
print(mha(torch.randn(2, 3, 64), torch.randn(2, 7, 64), torch.randn(2, 7, 64)).shape)   # cross-attention: [2, 3, 64] (query length wins)
print(sum(p.numel() for p in mha.parameters()))                    # expect 16640 = 4 layers x (64*64 + 64)

torch.Size([2, 4, 5, 16]) torch.Size([2, 4, 5, 5])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6464, 0.3536, 0.0000, 0.0000, 0.0000],
        [0.4229, 0.3327, 0.2443, 0.0000, 0.0000],
        [0.5032, 0.0516, 0.1476, 0.2976, 0.0000],
        [0.1328, 0.0983, 0.0639, 0.4939, 0.2111]])
torch.Size([2, 5, 64])
torch.Size([2, 3, 64])
16640


## 7. Feed-forward and the encoder layer

* **Feed-forward** - works on each word's vector on its own: `Linear(64→128) → ReLU → Dropout → Linear(128→64)`.
* **Residual connection** `x + sublayer(x)` keeps the original information and lets gradients flow back through deep stacks.
* **LayerNorm** rescales each word's 64 numbers to mean 0 / std 1, which keeps training stable.

Encoder layer = `LayerNorm(x + Dropout(SelfAttention(x)))` then `LayerNorm(x + Dropout(FeedForward(x)))`.

In [15]:
# ==== FeedForward ===============================================================
# WHAT : a small two-layer network applied to every word vector separately (same weights at every position).
# WHY  : attention MIXES information between words; this layer then PROCESSES each word (adds non-linearity).
# IN/OUT: [B, L, d_model] -> [B, L, d_model]
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)      # expand 64 -> 128
        self.linear2 = nn.Linear(d_ff, d_model)      # shrink 128 -> 64 (back to the model size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # inside-out: linear1 -> ReLU (negative numbers become 0 = the non-linearity) -> dropout -> linear2
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


# ==== EncoderLayer ==============================================================
# WHAT : one encoder block = self-attention + feed-forward, each wrapped in  LayerNorm(x + Dropout(block(x))).
# IN/OUT: [B, S, d_model] -> [B, S, d_model]  (the shape never changes, so layers can be stacked)
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)           # one LayerNorm after each block (each has its own learnable scale/shift)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        # block 1: every Hindi word looks at every other Hindi word (q = k = v = x).
        # "x + ..." is the RESIDUAL connection: the original x is added back, so nothing is lost and gradients flow easily.
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        # block 2: process each word's vector on its own
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x

In [16]:
# ---- Test: feed-forward + encoder layer ----------------------------------------
ff = FeedForward(config["d_model"], config["d_ff"], config["dropout"])
print(sum(p.numel() for p in ff.parameters()))           # expect 16576 = (64*128+128) + (128*64+64)

enc_layer = EncoderLayer(config["d_model"], config["num_heads"], config["d_ff"], config["dropout"])
print(sum(p.numel() for p in enc_layer.parameters()))    # expect 33472 = attention 16640 + ff 16576 + 2 LayerNorms x 128

x = pos_enc(emb(src))                                    # ids -> vectors + positions
out = enc_layer(x, src_mask)
print(out.shape)                                         # unchanged: [64, L, 64]
print(out.mean(dim=-1)[0, :3])                           # ~0 : LayerNorm made each word's 64 numbers average 0
print(out.std(dim=-1, unbiased=False)[0, :3])            # ~1 : ... and spread 1

16576
33472
torch.Size([64, 21, 64])
tensor([-1.8626e-08, -7.4506e-09,  3.7253e-09], grad_fn=<SliceBackward0>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)


## 8. Encoder, decoder and the full Transformer

* **Encoder** - embedding → positions → N encoder layers. Output: one rich vector per Hindi word.
* **Decoder layer** - 3 blocks: masked self-attention (`tgt_mask`), **cross-attention** (query = English side, key/value = encoder output, `src_mask`), feed-forward.
* **Transformer** - encoder + decoder + a final `Linear(d_model → English vocab)` that gives **logits** (raw scores per word).

The model returns *logits*, not probabilities: `CrossEntropyLoss` applies the softmax itself.
`nn.ModuleList` is required so PyTorch registers (and trains) every layer.

In [17]:
# ==== Encoder ===================================================================
# WHAT : reads the Hindi sentence and produces a context-aware vector for every Hindi word.
# FLOW : ids -> embedding -> + positions -> EncoderLayer x num_layers
# IN   : src [B,S], src_mask [B,1,1,S]      OUT: enc_out [B,S,d_model]
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        # nn.ModuleList registers every layer so its weights are found by model.parameters() and trained.
        # [... for _ in range(n)] builds n SEPARATE layers, each with its own weights (the _ just means "unused counter").
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, src, src_mask):
        x = self.pos_enc(self.embedding(src))       # ids -> vectors, then add the position information
        for layer in self.layers:                   # run through the layers one after another
            x = layer(x, src_mask)                  # each layer's output is the next layer's input
        return x


# ==== DecoderLayer ==============================================================
# WHAT : one decoder block with THREE sub-blocks:
#        1. masked self-attention : English words look at EARLIER English words only
#        2. cross-attention       : English words look at the Hindi sentence (the encoder output)
#        3. feed-forward          : per-word processing
# IN   : x [B,T,d], enc_out [B,S,d], src_mask [B,1,1,S], tgt_mask [B,1,T,T]     OUT: [B,T,d]
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)     # same class as the encoder's ...
        self.cross_attn = MultiHeadAttention(d_model, num_heads)    # ... but a different instance = different weights
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        # 1. q = k = v = x (English side), tgt_mask hides padding AND future words
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        # 2. q = English side, k = v = enc_out (Hindi side): "which Hindi words matter for the word I am writing?"
        #    src_mask hides the Hindi <PAD> positions. Output length follows the QUERY (English).
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        # 3. per-word feed-forward
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


# ==== Decoder ===================================================================
# WHAT : a stack of DecoderLayers that turns the (shifted) English words + the Hindi context into English-side vectors.
# IN   : tgt_in [B,T], enc_out [B,S,d], src_mask, tgt_mask      OUT: [B,T,d_model]
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout, max_len, pad_id):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model, pad_id)    # English vocabulary
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, tgt_in, enc_out, src_mask, tgt_mask):
        x = self.pos_enc(self.embedding(tgt_in))
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, tgt_mask)    # EVERY layer sees the same enc_out
        return x


# ==== Transformer ===============================================================
# WHAT : the whole model: encoder + decoder + output layer.
# IN   : src [B,S], tgt_in [B,T]            OUT: logits [B,T,V_english]
# NOTE : returns raw scores (logits), NOT probabilities. nn.CrossEntropyLoss does the softmax itself,
#        so applying softmax here as well would be a bug. Call torch.softmax(logits, -1) only when you want probabilities.
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model, num_heads, d_ff, num_layers,
                 dropout, max_len, src_pad_id, tgt_pad_id):
        super().__init__()
        self.src_pad_id = src_pad_id                 # remembered so forward() can build the masks itself
        self.tgt_pad_id = tgt_pad_id
        self.encoder = Encoder(src_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, src_pad_id)
        self.decoder = Decoder(tgt_vocab, d_model, num_heads, d_ff, num_layers, dropout, max_len, tgt_pad_id)
        self.output_layer = nn.Linear(d_model, tgt_vocab)    # 64 numbers -> one score for each English word

    def forward(self, src, tgt_in):
        src_mask = make_pad_mask(src, self.src_pad_id)              # hide Hindi padding
        tgt_mask = make_tgt_mask(tgt_in, self.tgt_pad_id)           # hide English padding + the future
        enc_out = self.encoder(src, src_mask)                       # step 1: understand the Hindi sentence
        dec_out = self.decoder(tgt_in, enc_out, src_mask, tgt_mask) # step 2: build English-side vectors, looking at the Hindi
        return self.output_layer(dec_out)                           # step 3: vectors -> scores over the English vocabulary


# ==== build_model ===============================================================
# WHAT : the ONE place that creates a Transformer from a config dict + the two vocabularies.
# WHY  : used by both training and checkpoint loading, so the two can never disagree on the architecture.
def build_model(config, src_stoi, tgt_stoi):
    return Transformer(len(src_stoi), len(tgt_stoi), config["d_model"], config["num_heads"],
                       config["d_ff"], config["num_layers"], config["dropout"], config["max_len"],
                       src_stoi['<PAD>'], tgt_stoi['<PAD>'])

In [18]:
# ---- Test: the full forward pass up to softmax ---------------------------------
test_model = build_model(config, src_stoi, tgt_stoi)
logits = test_model(src, tgt_in)                             # forward pass: ids in, scores out
print(logits.shape)                                          # expect [64, L, 98]: a score for each English word at each position
print(sum(p.numel() for p in test_model.parameters()))       # expect 184994 weights in total

probs = torch.softmax(logits, dim=-1)                        # scores -> probabilities (what a "forward pass up to softmax" means)
print(probs.sum(dim=-1)[0, :3])                              # all ones
print(probs[0, 0].max().item())                              # small (~0.01-0.03): an untrained model is near-uniform (1/98 = 0.0102)

# Causal check: changing a FUTURE word must not change earlier predictions.
test_model.eval()                                            # dropout off -> repeatable results
with torch.no_grad():                                        # no gradient bookkeeping needed for a check
    l1 = test_model(src, tgt_in)
    tgt2 = tgt_in.clone()                                    # clone() = an independent copy we can edit
    tgt2[:, 5] = 7                                           # overwrite the word at position 5 in every sentence
    l2 = test_model(src, tgt2)
print(torch.allclose(l1[:, :5], l2[:, :5], atol=1e-5))       # True : positions 0-4 cannot see position 5
print(torch.allclose(l1[:, 5:], l2[:, 5:], atol=1e-5))       # False: positions 5+ do change

torch.Size([64, 16, 450])
254338
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SliceBackward0>)
0.010749764740467072
True
False


## 9. Loss, backward pass and Adam: one training step

```
forward -> loss -> zero_grad -> backward -> clip -> optimizer.step
```

* **Cross-entropy loss** = `-log(probability given to the correct word)`, averaged over positions. `ignore_index` skips `<PAD>` positions.
  An untrained model scores about `ln(vocab) ≈ ln(98) ≈ 4.6`.
* **`loss.backward()`** computes a *gradient* (how the loss changes if the weight changes) for every weight.
* **`zero_grad()`** first, otherwise gradients pile up across batches.
* **Gradient clipping** caps huge gradients so one bad batch cannot wreck training.
* **Adam** gives every weight its own step size using running averages of its gradient.

In [19]:
# ==== compute_loss ==============================================================
# WHAT : how wrong were the predictions? One number (lower = better).
# HOW  : criterion = nn.CrossEntropyLoss. For every position it applies log-softmax to the 98 scores and takes
#        -log(probability of the CORRECT word), then averages. Positions whose target is <PAD> are skipped (ignore_index).
# IN   : logits [B,T,V], tgt_out [B,T]
def compute_loss(logits, tgt_out, criterion):
    V = logits.size(-1)                                   # size of the English vocabulary (98)
    # CrossEntropyLoss wants [N, classes] and [N]. reshape(-1, V) means "V columns, work out the rows":
    # [B, T, V] -> [B*T, V]  (every word position in the batch becomes one row).  [B, T] -> [B*T].
    return criterion(logits.reshape(-1, V), tgt_out.reshape(-1))


# ==== training_step =============================================================
# WHAT : ONE learning step on ONE batch: forward -> loss -> backward -> weight update.
# OUT  : the loss as a plain Python float (for printing / averaging)
def training_step(model, batch, criterion, optimizer, unk_prob=0.03):
    src, tgt_in, tgt_out = batch                                   # unpack the three tensors from make_batch
    drop = (torch.rand(src.shape) < unk_prob) & (src > 3)          # ~3% of real source words (ids 0-3 are special tokens)
    src = src.masked_fill(drop, 3)                                 # ... become <UNK> (id 3), so the model learns to handle unknown words
    model.train()                                                  # training mode: dropout ON
    logits = model(src, tgt_in)                                    # 1. forward pass (also records the maths for backward)
    loss = compute_loss(logits, tgt_out, criterion)                # 2. one number: how wrong were we?
    optimizer.zero_grad()                                          # 3. clear last step's gradients (backward ADDS to .grad)
    loss.backward()                                                # 4. backward pass: fills p.grad with d(loss)/d(weight) for every weight
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)     #    if all gradients together are longer than 1.0, scale them down
    optimizer.step()                                               # 5. Adam moves every weight a small step against its gradient
    return loss.item()                                             # .item(): a 1-element tensor -> a normal Python number

In [20]:
# ---- Test: first loss, then overfit ONE batch (proves backward + Adam work) -----
tmp_model = build_model(config, src_stoi, tgt_stoi)                # throw-away model, so the real training starts clean
tmp_criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
# Adam: gives each weight its own step size from running averages of its gradient (betas) ; eps avoids dividing by ~0
tmp_optimizer = torch.optim.Adam(tmp_model.parameters(), lr=config["lr"], betas=(0.9, 0.98), eps=1e-9)

one_batch = make_batch(train_pairs[:64], src_stoi, tgt_stoi)       # the SAME 64 sentences every step
for step in range(100):
    loss = training_step(tmp_model, one_batch, tmp_criterion, tmp_optimizer)
    if step % 20 == 0:
        print(step, round(loss, 4))        # starts near 4.6 (= ln 98), then falls steadily: the model is learning

0 6.2623
20 4.8506
40 3.8936
60 3.1315
80 2.5616


## 10. Epochs, validation and checkpoints

* **Step** = one batch. **Epoch** = enough steps to see the whole training set once.
* `train_one_epoch` learns; `evaluate` only measures (`model.eval()`, `torch.no_grad()`, no weight updates).
* Read the two numbers together: both falling = learning. Train falling while validation rises = **overfitting**.
* The best-validation model is saved as a **checkpoint** that also stores the vocab and config, so generation works later without retraining.

In [21]:
# ==== train_one_epoch ===========================================================
# WHAT : one full pass over `pairs`, learning on every batch.
# OUT  : the average training loss of the epoch
def train_one_epoch(model, pairs, batch_size, src_stoi, tgt_stoi, criterion, optimizer):
    losses = []
    for batch in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=True):    # shuffle: new batch order every epoch
        losses.append(training_step(model, batch, criterion, optimizer))               # learn, and remember this batch's loss
    return sum(losses) / len(losses)                                                   # mean over all batches


# ==== evaluate ==================================================================
# WHAT : average loss on data the model is NOT learning from (validation or test).
# DIFF vs training: dropout off, no gradients, no weight updates, fixed order.
def evaluate(model, pairs, batch_size, src_stoi, tgt_stoi, criterion):
    model.eval()                                   # evaluation mode: dropout OFF -> repeatable measurement
    losses = []
    with torch.no_grad():                          # do not record gradients here: faster, less memory
        for src, tgt_in, tgt_out in get_batches(pairs, batch_size, src_stoi, tgt_stoi, shuffle=False):
            logits = model(src, tgt_in)
            losses.append(compute_loss(logits, tgt_out, criterion).item())
    return sum(losses) / len(losses)


# ==== save_checkpoint ===========================================================
# WHAT : writes ONE file holding everything needed to use the model later.
# WHY  : weights alone are useless without the vocab (id <-> word) and the sizes the model was built with.
def save_checkpoint(path, model, src_itos, tgt_itos, config):
    torch.save({"model_state": model.state_dict(),      # state_dict() = every learned weight, as {layer name: tensor}
                "src_itos": src_itos, "tgt_itos": tgt_itos,   # only the lists are saved; stoi is rebuilt from them
                "config": config}, path)


# ==== load_checkpoint ===========================================================
# WHAT : rebuilds the exact model that was saved. OUT: (model, src_stoi, tgt_stoi, tgt_itos)
def load_checkpoint(path):
    ckpt = torch.load(path)                              # read the dictionary back from disk
    if "model_state" not in ckpt:                        # an old file that holds only weights
        raise ValueError(f"{path} is an old weights-only file. Delete it, or set config['retrain'] = True.")
    src_itos, tgt_itos, saved_config = ckpt["src_itos"], ckpt["tgt_itos"], ckpt["config"]
    src_stoi = {tok: i for i, tok in enumerate(src_itos)}      # rebuild token -> id from the saved id -> token list
    tgt_stoi = {tok: i for i, tok in enumerate(tgt_itos)}
    model = build_model(saved_config, src_stoi, tgt_stoi)      # same architecture as when it was saved ...
    model.load_state_dict(ckpt["model_state"])                 # ... then pour the saved weights into it
    model.eval()                                               # ready for generation (dropout off)
    return model, src_stoi, tgt_stoi, tgt_itos


# ==== train_model ===============================================================
# WHAT : the whole training run: for each epoch -> learn on train, measure on validation, keep the best model.
# OUT  : the model after the LAST epoch (the BEST one is in the checkpoint file)
def train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config):
    src_stoi, src_itos = src_vocab                 # each vocab is the (stoi, itos) pair from build_vocab_from_sentences
    tgt_stoi, tgt_itos = tgt_vocab

    model = build_model(config, src_stoi, tgt_stoi)                          # brand-new random weights every call
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])          # <PAD> targets do not count
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], betas=(0.9, 0.98), eps=1e-9)

    best_val = float("inf")                        # infinity: the first validation loss is always "better"
    for epoch in range(1, config["num_epochs"] + 1):
        train_loss = train_one_epoch(model, train_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion, optimizer)
        val_loss = evaluate(model, val_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion)
        print(f"epoch {epoch:2d} | train {train_loss:.4f} | val {val_loss:.4f}")    # :2d / :.4f = number formatting
        if val_loss < best_val:                    # new best on UNSEEN data -> save it
            best_val = val_loss
            save_checkpoint(config["checkpoint_path"], model, src_itos, tgt_itos, config)
    return model

## 11. Generation and final test

**Autoregressive greedy decoding** - build the English sentence one word at a time: run the decoder on the words so far,
take the scores of the *last* position only, pick the highest, append it, repeat until `<EOS>`.
The encoder runs once; the Hindi side never changes.

**Exact match** - the share of sentences translated word for word like the reference (a stricter score than loss).

In [22]:
# ==== translate =================================================================
# WHAT : Hindi sentence (str) -> English sentence (str).
# HOW  : "autoregressive greedy decoding" - write one word, feed it back in, write the next, until <EOS>.
#        1. encoder reads the Hindi ONCE   2. start with [<SOS>]   3. repeat: decoder -> scores of the LAST position
#        -> pick the highest-scoring word (argmax = greedy) -> append -> stop at <EOS> or max_len
def translate(model, sentence, src_stoi, tgt_stoi, tgt_itos, max_len=20):
    model.eval()                                                    # dropout off
    pe_len = model.encoder.pos_enc.pe.size(1)                       # how many positions the model has encodings for
    src_ids = encode_sentence(sentence, src_stoi)
    unknown = [w for w, i in zip(tokenize_text_line(sentence), src_ids[1:-1]) if i == src_stoi['<UNK>']]
    if unknown:
        print('warning: not in vocabulary ->', unknown)
    if len(src_ids) > pe_len:                                       # too long -> would crash with an obscure shape error
        raise ValueError(f"sentence has {len(src_ids)} tokens; model supports {pe_len}. Raise config['max_len'].")
    max_len = min(max_len, pe_len - 1)                              # never generate past the positional table

    with torch.no_grad():                                           # no gradients while generating
        src = torch.tensor([src_ids])                               # the EXTRA [ ] adds a batch dimension: [S] -> [1, S]
        src_mask = make_pad_mask(src, src_stoi['<PAD>'])
        enc_out = model.encoder(src, src_mask)                      # the Hindi is understood ONCE and reused every step

        tgt = torch.tensor([[tgt_stoi['<SOS>']]])                   # decoder input so far: just <SOS>, shape [1, 1]
        for _ in range(max_len):
            tgt_mask = make_tgt_mask(tgt, tgt_stoi['<PAD>'])        # rebuilt each time because tgt grows by one word
            dec_out = model.decoder(tgt, enc_out, src_mask, tgt_mask)   # [1, t, d_model]
            # dec_out[:, -1] = the LAST position only (that one predicts the NEXT word) -> [1, d_model] -> scores [1, V]
            next_logits = model.output_layer(dec_out[:, -1])
            next_id = torch.argmax(next_logits, dim=-1)             # greedy: the single highest-scoring word -> shape [1]
            tgt = torch.cat([tgt, next_id.view(1, 1)], dim=1)       # view(1,1) -> [1,1]; cat along dim 1 appends it to the sentence
            if next_id.item() == tgt_stoi['<EOS>']:                 # .item(): 1-element tensor -> Python int; <EOS> = "I am done"
                break
    return decode_sequence(tgt[0].tolist(), tgt_itos)               # [0] = the only sentence; tolist() -> plain ids -> words


# ==== exact_match_accuracy ======================================================
# WHAT : the fraction of sentences translated EXACTLY like the reference (stricter and easier to read than loss).
# OUT  : (accuracy between 0 and 1, list of (hindi, reference, model_output) for every mistake)
def exact_match_accuracy(model, pairs, src_stoi, tgt_stoi, tgt_itos):
    correct = 0
    mistakes = []
    for hi, en in pairs:
        out = translate(model, hi, src_stoi, tgt_stoi, tgt_itos)
        ref = " ".join(tokenize_text_line(en))                 # tokenize + join so the reference is spaced like the output
        if tokenize_text_line(out) == tokenize_text_line(ref):
            correct += 1
        else:
            mistakes.append((hi, ref, out))                    # keep failures so you can read them
    return correct / len(pairs), mistakes

## 12. `main()`: train (if needed) → test → return the model

`main()` **never waits for keyboard input**, so it always finishes.
It trains only when there is no checkpoint (or `config["retrain"]` is `True`), prints the test loss and exact-match score,
shows a few translations, and returns the loaded model so the next cell can use it.

In [23]:
# ==== main ======================================================================
# WHAT : the complete pipeline in order: data -> vocab -> (train) -> load best -> test -> examples.
# OUT  : (model, src_stoi, tgt_stoi, tgt_itos) so the cells below can translate with it.
# NOTE : there is NO input() here, so it always finishes (an input box can sit hidden in some editors).
def main():
    random.seed(config["seed"])                          # repeatable shuffling ...
    torch.manual_seed(config["seed"])                    # ... and repeatable initial weights

    # 1. data -> splits -> vocab. The vocab is built from the TRAIN split only (val / test must stay unseen).
    pairs = load_pairs_from_txt(config["data_path"])
    train_pairs, val_pairs, test_pairs = split_data_test_train(pairs, seed=config["seed"])
    src_vocab = build_vocab_from_sentences([p[0] for p in train_pairs])    # (stoi, itos) for Hindi
    tgt_vocab = build_vocab_from_sentences([p[1] for p in train_pairs])    # (stoi, itos) for English
    print(f"data: {len(train_pairs)} train / {len(val_pairs)} val / {len(test_pairs)} test | "
          f"vocab: {len(src_vocab[0])} Hindi, {len(tgt_vocab[0])} English")

    # 2. train only when needed: forced by config, or there is nothing saved yet
    if config["retrain"] or not os.path.exists(config["checkpoint_path"]):
        train_model(train_pairs, val_pairs, src_vocab, tgt_vocab, config)
    else:
        print(f"found {config['checkpoint_path']} - skipping training (set config['retrain'] = True to retrain)")

    # 3. always use the BEST saved model (lowest validation loss), not just whatever the last epoch produced
    model, src_stoi, tgt_stoi, tgt_itos = load_checkpoint(config["checkpoint_path"])

    # 4. the final exam on the untouched test split
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_stoi['<PAD>'])
    test_loss = evaluate(model, test_pairs, config["batch_size"], src_stoi, tgt_stoi, criterion)
    eval_pairs = test_pairs[:1000]                       # decoding one sentence at a time is slow: 1000 is enough
    acc, mistakes = exact_match_accuracy(model, eval_pairs, src_stoi, tgt_stoi, tgt_itos)
    print(f"test loss: {test_loss:.4f} | exact match: {acc:.2%} ({len(eval_pairs) - len(mistakes)}/{len(eval_pairs)})")
    for hi, ref, out in mistakes[:5]:                    # show up to 5 failures (none if the score is 100%)
        print("  HI :", hi, "\n  REF:", ref, "\n  OUT:", out)

    # 5. a few examples to read with your own eyes
    for hi, en in test_pairs[:3]:
        print("HI :", hi, "\nREF:", en, "\nOUT:", translate(model, hi, src_stoi, tgt_stoi, tgt_itos), "\n")

    return model, src_stoi, tgt_stoi, tgt_itos

In [24]:
# ---- Run the whole pipeline (finishes on its own; no input box) ----------------
# First run: trains and saves best_model.pt. Later runs: finds the file and skips training.
model, src_stoi, tgt_stoi, tgt_itos = main()

data: 80000 train / 10000 val / 10000 test | vocab: 451 Hindi, 450 English
epoch  1 | train 0.9341 | val 0.0698
epoch  2 | train 0.0896 | val 0.0134
epoch  3 | train 0.0454 | val 0.0030
epoch  4 | train 0.0341 | val 0.0013
epoch  5 | train 0.0299 | val 0.0008
epoch  6 | train 0.0273 | val 0.0006
epoch  7 | train 0.0254 | val 0.0005
epoch  8 | train 0.0249 | val 0.0007
epoch  9 | train 0.0245 | val 0.0005
epoch 10 | train 0.0238 | val 0.0004
epoch 11 | train 0.0230 | val 0.0004
epoch 12 | train 0.0223 | val 0.0006
epoch 13 | train 0.0224 | val 0.0003
epoch 14 | train 0.0219 | val 0.0003
epoch 15 | train 0.0221 | val 0.0003
epoch 16 | train 0.0212 | val 0.0003
epoch 17 | train 0.0219 | val 0.0002
epoch 18 | train 0.0208 | val 0.0003
epoch 19 | train 0.0210 | val 0.0002
epoch 20 | train 0.0206 | val 0.0003
test loss: 0.0002 | exact match: 99.80% (998/1000)
  HI : मोहन, क्या आप कृपया मुझे आज का कैलेंडर आज शाम तक दिखा सकते हैं? 
  REF: Mohan , could you please show me today's calendar by th

## 13. Try your own sentences

Edit the strings and re-run. Rules for good queries:
* Use only words from the training data (about 77 Hindi words); any other word becomes `<UNK>` and the output is unreliable.
* The danda may be attached (`है।`) or spaced (`है ।`); both work.
* Recombinations of known words that were never seen together are a fair test that the model learned the grammar.

In [25]:
# ==== ask =======================================================================
# WHAT : translate one Hindi sentence with the trained model (the globals from main()) and print both sides.
def ask(sentence):
    print("HI :", sentence)
    print("EN :", translate(model, sentence, src_stoi, tgt_stoi, tgt_itos), "\n")


ask("लड़की रोज़ आम खाती है।")                   # danda attached to the last word
ask("वे हर शाम बगीचे में दूध पीते हैं ।")        # a recombination of known words
ask("शिक्षक रविवार को स्कूल में कहानी लिखता है ।")

HI : लड़की रोज़ आम खाती है।
EN : The girl eats a mango every day. 

HI : वे हर शाम बगीचे में दूध पीते हैं ।
EN : They drink milk in the garden every evening. 

HI : शिक्षक रविवार को स्कूल में कहानी लिखता है ।
EN : The teacher writes a story at school on Sunday. 



In [26]:
# ==== chat (optional; TERMINAL / .py script only) ===============================
# WHAT : a type-a-sentence loop. input() waits for the keyboard, which is why it is NOT inside main().
# In a notebook use ask("...") above instead. To try it in a terminal, call chat() yourself.
def chat():
    while True:
        q = input("Hindi> ").strip()          # wait for a typed line; strip() removes surrounding spaces
        if q in ("", "quit"):                 # empty line or the word quit -> leave the loop
            break
        ask(q)